# Testing LangChain Integration with TurboQuant

This notebook demonstrates how to use the `TurboQuantVectorStore` as a drop-in replacement for standard LangChain vector stores (like FAISS or Chroma). 

We will use a lightweight embedding model from `sentence-transformers` via `langchain-huggingface`.

In [ ]:
# Install the necessary LangChain packages for the test
!uv pip install langchain-huggingface langchain-community

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from tqtorch.search.langchain import TurboQuantVectorStore

In [ ]:
# 1. Initialize the embedding model
# 'all-MiniLM-L6-v2' is a fast, 384-dimensional embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [ ]:
# 2. Prepare sample documents
docs = [
    "The quick brown fox jumps over the lazy dog.",
    "TurboQuant scalar-quantizes coordinates optimally using standard precomputed Gaussian codebooks.",
    "LangChain makes building LLM applications and RAG pipelines incredibly easy.",
    "PyTorch provides native support for GPU-accelerated tensor operations.",
    "Vector databases are used to store high-dimensional embeddings for semantic search.",
    "Apples, bananas, and oranges are types of fruit."
]

metadatas = [{"doc_id": i} for i in range(len(docs))]

# 3. Build the TurboQuant VectorStore
# We'll use 4-bit per coordinate quantization (Algorithm 2: MSE + QJL for inner products) 
vector_store = TurboQuantVectorStore.from_texts(
    texts=docs,
    embedding=embeddings,
    metadatas=metadatas,
    bits=4,
    metric="ip"  # inner product
)

print(f"Successfully ingested {len(docs)} documents into TurboQuantIndex.")

In [ ]:
# 4. Perform a semantic similarity search
query = "How does TurboQuant compress data?"

print(f"Query: '{query}'\n")

# Return top 2 most relevant documents
results = vector_store.similarity_search(query, k=2)

for i, result in enumerate(results):
    print(f"Result {i+1} (Metadata: {result.metadata}):")
    print(f"Content: {result.page_content}\n")